# CS 370 Project Two: Pirate Intelligent Agent

**Student Name:** CJ Ralls  
**Course:** CS 370  

This notebook completes the Q-training portion of the treasure hunt game. The pirate agent uses deep Q-learning to learn a path through the maze and reach the treasure. The provided `.py` files, such as `TreasureMaze.py` and `GameExperience.py`, should not be modified.

In [ ]:
# Import required libraries
import numpy as np
import random
from time import time, sleep

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from TreasureMaze import TreasureMaze
from GameExperience import GameExperience

print("TensorFlow version:", tf.__version__)

## Maze Environment

The maze is represented as a matrix where valid spaces and blocked spaces define the game environment. The pirate agent learns which actions move it closer to the treasure by interacting with this environment.

In [ ]:
# This maze layout is commonly used in the CS 370 treasure hunt project.
# If your starter notebook already includes a maze variable, you may keep the starter version instead.
maze = np.array([
    [1., 0., 1., 1., 1., 1., 1., 1.],
    [1., 0., 1., 1., 1., 0., 1., 1.],
    [1., 1., 1., 1., 0., 1., 0., 1.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 0., 1., 1., 1., 1., 1.],
    [1., 1., 1., 0., 1., 0., 0., 0.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 1., 1., 0., 1., 1., 1.]
])

qmaze = TreasureMaze(maze)
print("Maze created with shape:", maze.shape)

## Build the Deep Q-Learning Model

The neural network estimates Q-values for each possible action. These values represent the expected future reward for taking an action from the current state.

In [ ]:
def build_model(maze):
    """Builds and compiles the deep Q-learning neural network.

    The input size is based on the flattened maze state. The output layer has
    four nodes because the pirate can move in four possible directions.
    """
    input_size = maze.size
    action_size = 4

    model = keras.Sequential([
        layers.Input(shape=(input_size,)),
        layers.Dense(input_size, activation="relu"),
        layers.Dense(input_size, activation="relu"),
        layers.Dense(action_size, activation="linear")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse"
    )

    return model

model = build_model(maze)
model.summary()

## Training Helper

The `train_step` function keeps model training organized. It receives the training inputs and target Q-values from experience replay and updates the neural network.

In [ ]:
@tf.function
def train_step(inputs, targets):
    """Runs one training step for the Q-network."""
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss = tf.keras.losses.mean_squared_error(targets, predictions)
        loss = tf.reduce_mean(loss)
    gradients = tape.gradient(loss, model.trainable_variables)
    model.optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

## Q-Training Algorithm

This is the main section of the project. The pirate agent explores the maze, stores experiences, trains from replay memory, and gradually shifts from exploration to exploitation.

In [ ]:
def qtrain(model, maze, **opt):
    """Train the pirate agent using deep Q-learning.

    Parameters are passed through opt so the training setup can be adjusted
    without rewriting the algorithm. The function uses experience replay to
    train the model from past episodes.
    """
    # Hyperparameters
    n_epoch = opt.get("n_epoch", 15000)
    max_memory = opt.get("max_memory", 1000)
    data_size = opt.get("data_size", 50)
    target_update_freq = opt.get("target_update_freq", 50)
    discount = opt.get("discount", 0.95)
    epsilon = opt.get("epsilon", 1.0)
    epsilon_min = opt.get("epsilon_min", 0.05)
    epsilon_decay = opt.get("epsilon_decay", 0.995)

    # Environment and experience replay setup
    qmaze = TreasureMaze(maze)
    experience = GameExperience(model, max_memory=max_memory)

    start_time = time()
    win_history = []
    hsize = qmaze.maze.size // 2

    for epoch in range(n_epoch):
        # Randomly select a valid free cell as the starting point.
        free_cells = qmaze.free_cells
        agent_cell = random.choice(free_cells)
        qmaze.reset(agent_cell)

        envstate = qmaze.observe()
        n_episodes = 0
        loss = 0.0
        game_over = False

        while not game_over:
            previous_envstate = envstate

            # Choose an action using epsilon-greedy behavior.
            if np.random.rand() < epsilon:
                action = random.choice(qmaze.valid_actions())
            else:
                q_values = model.predict(previous_envstate, verbose=0)
                action = int(np.argmax(q_values[0]))

            # Apply the action to the maze environment.
            envstate, reward, game_status = qmaze.act(action)

            if game_status == "win":
                game_over = True
                win_history.append(1)
            elif game_status == "lose":
                game_over = True
                win_history.append(0)

            # Store the episode for experience replay.
            episode = [previous_envstate, action, reward, envstate, game_over]
            experience.remember(episode)
            n_episodes += 1

            # Train only when there are enough stored experiences.
            inputs, targets = experience.get_data(data_size=data_size)
            inputs = tf.convert_to_tensor(inputs, dtype=tf.float32)
            targets = tf.convert_to_tensor(targets, dtype=tf.float32)
            loss = train_step(inputs, targets).numpy()

        # Keep only recent history for measuring completion.
        if len(win_history) > hsize:
            win_history = win_history[-hsize:]

        win_rate = sum(win_history) / len(win_history) if win_history else 0

        # Gradually reduce exploration so the agent uses learned behavior more often.
        if epsilon > epsilon_min:
            epsilon *= epsilon_decay
            epsilon = max(epsilon_min, epsilon)

        # Print progress in a readable format.
        template = "Epoch: {:03d}/{:d} | Loss: {:.4f} | Episodes: {:d} | Win count: {:d} | Win rate: {:.3f} | Epsilon: {:.3f}"
        print(template.format(epoch + 1, n_epoch, loss, n_episodes, sum(win_history), win_rate, epsilon))

        # Stop training once the agent is consistently successful.
        if win_rate > 0.9 and len(win_history) >= hsize:
            print("Training complete. The pirate agent has reached the target win rate.")
            break

    total_time = time() - start_time
    print("Training time: {:.2f} seconds".format(total_time))
    return total_time

## Train the Model

This cell may take several minutes to run in Codio. If training is slow, keep the same algorithm but lower `n_epoch` while testing, then raise it for the final run.

In [ ]:
model = build_model(maze)
qtrain(
    model,
    maze,
    n_epoch=1000,
    max_memory=8 * maze.size,
    data_size=32,
    target_update_freq=50
)

## Completion Check

Run the required course completion check after training. This confirms whether the pirate agent can solve the maze successfully.

In [ ]:
# Run these cells if they are included in your starter notebook environment.
# They are usually provided by SNHU/Codio as part of the assignment.
try:
    completion_check(model, qmaze)
    show(qmaze)
except NameError:
    print("completion_check or show is not defined in this environment. Run this cell inside the Codio starter notebook.")

## Notes for Submission

Save this notebook with your name in the file name, then export it as an HTML file for submission. The project guidelines require the completed Jupyter Notebook to be downloaded as an HTML file and submitted along with the design defense document.